# Surface soak vs near-surface data quality (P45_06 vs P45_07)

Does holding the CTD at the surface (the soak) actually give cleaner near-surface data? This reads the
L1 cnv for every cast in both cruises and, for each cast, measures two things straight from the data:

- **Soak time (s):** how long the package stayed in the water near the surface (above a few db) before
  the clean descent began. This is the *operational* fact, did they soak or not, read from the data,
  not from the log sheet.
- **Near-surface spiking:** the high-frequency noise in salinity (and temperature) over the first
  ~10 db of the clean descent. This is the *data-quality effect*. A well-equilibrated cast (pump
  primed, cell flushed, thermistor settled) has a quiet top-of-descent; an unequilibrated one spikes.

It then compares the two cruises, and, more importantly, **pools every cast from both cruises and
tests soak time against near-surface spiking directly** (a per-cast relationship, which is stronger
than a cruise-vs-cruise label because it does not rest on the two cruises being otherwise identical).

**Read this honestly (the limits matter):**
- This is **observational, not a controlled test.** P45_06 and P45_07 differ in season, sea state and
  crew, not only in soak. Rougher weather tends to mean both less soak and more heave, so soak time
  and sea state are tangled. The clean proof would be alternating soak / no-soak casts within one
  cruise, same conditions.
- Metrics are computed on **L1 (pre-alignment) data** so both cruises are judged at the same stage.
  Some salinity spiking at L1 comes from the conductivity/temperature time-offset at sharp gradients,
  not from the soak; the short-window residual used here suppresses smooth gradients but cannot remove
  all of it. A publication-grade version would measure spiking on aligned + cell-thermal-mass data and
  add the near-surface loop-edit flag fraction.
- Treat every result as an **association**, phrased as "casts that soaked longer tended to have a
  quieter near-surface," never "the soak caused it."

Sources: Sea-Bird Data Processing manual and SBE deployment notes (surface soak to prime the pump and
equilibrate sensors); GO-SHIP/WOCE CTD practice (discard the unequilibrated top of the descent);
UNESCO 1983 (PSS-78 salinity).

## 1. Imports, config, helpers

In [ ]:
from __future__ import annotations
import re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import rcParams
try:
    from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image = None

# --------------------------------------------------------------- config
CTD_ROOT = Path(r"C:\Projects\ctd_pipeline")
# (cruise_id, label) - label is how it appears on the plots
CRUISES = [("P45_06", "soak"), ("P45_07", "no soak")]

SOAK_MAX_DB  = 5.0     # "near surface" band: in-water scans above this count toward soak dwell
NEARSURF_DB  = 10.0    # window at the top of the clean descent used for the spiking metric
SPIKE_WIN    = 5       # rolling-median window (scans) for the high-frequency residual
MIN_DESC_SCANS = 10    # need at least this many scans in the near-surface window to score a cast
C_STD = 4.2914
SECTION_DEFAULT_FS = 24.0
SECTION_SOAK_MIN_DROP_DB = 0.10

sys.path.insert(0, str(CTD_ROOT))
try:
    from ctd_lib import practical_salinity
    _HAVE_LIB = True
except Exception as e:
    _HAVE_LIB = False
    def practical_salinity(C_Sm, T90, P):
        a=[0.0080,-0.1692,25.3851,14.0941,-7.0261,2.7081]; b=[0.0005,-0.0056,-0.0066,-0.0375,0.0636,-0.0144]
        c=[0.6766097,2.00564e-2,1.104259e-4,-6.9698e-7,1.0031e-9]; d=[3.426e-2,4.464e-4,4.215e-1,-3.107e-3]
        e=[2.070e-5,-6.370e-10,3.989e-15]
        R=np.asarray(C_Sm,float)/C_STD; T68=np.asarray(T90,float)*1.00024
        Rp=1+(P*(e[0]+e[1]*P+e[2]*P*P))/(1+d[0]*T68+d[1]*T68*T68+(d[2]+d[3]*T68)*R)
        rt=c[0]+c[1]*T68+c[2]*T68**2+c[3]*T68**3+c[4]*T68**4; Rt=np.maximum(R/(Rp*rt),1e-12); s=np.sqrt(Rt)
        dS=(T68-15)/(1+0.0162*(T68-15))*(b[0]+b[1]*s+b[2]*Rt+b[3]*Rt*s+b[4]*Rt**2+b[5]*Rt**2*s)
        return a[0]+a[1]*s+a[2]*Rt+a[3]*Rt*s+a[4]*Rt**2+a[5]*Rt**2*s+dS

try:
    from scipy.stats import mannwhitneyu, spearmanr
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

rcParams.update({"figure.dpi": 120, "savefig.dpi": 200, "font.size": 9, "axes.titlesize": 10,
                 "axes.labelsize": 9, "axes.linewidth": 0.8, "xtick.direction": "in",
                 "ytick.direction": "in", "legend.frameon": True, "font.family": "DejaVu Sans"})
CRUISE_COLOR = {}
_PAL = ["#2E6FB0", "#E8703A", "#4C9F70", "#9467BD"]


def new_static_fig(nrows=1, ncols=1, **kw):
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(**kw); FigureCanvasAgg(fig)
    return fig, fig.subplots(nrows, ncols)


def finish(fig, out):
    try: fig.savefig(out); ok = True
    except Exception as ex: print("savefig failed:", ex); ok = False
    if ok and Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: display(fig)
    return out if ok else None


def _despine(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)


# --------------------------------------------------------------- cnv readers
def parse_cnv_raw(path):
    if path is None: return None
    text = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, ln in enumerate(text):
        s = ln.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(ln)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    if ds is None: return None
    cols = [names[k] for k in sorted(names)]
    rows = [l.split()[:len(cols)] for l in text[ds:] if l.strip() and len(l.split()) >= len(cols)]
    df = pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
    bad = (df.le(-8e-29) & df.ge(-1.2e-28)) | (df.abs() > 1e30)
    if "flag" in bad.columns: bad["flag"] = False
    return df.mask(bad)


def _sample_rate(path):
    for ln in Path(path).read_text(encoding="latin-1", errors="replace").splitlines():
        low = ln.lower()
        if "interval" in low and "seconds" in low:
            m = re.search(r"seconds\s*:\s*([0-9.eE+-]+)", ln)
            if m:
                try:
                    dt = float(m.group(1))
                    if dt > 0: return 1.0 / dt
                except ValueError: pass
    return SECTION_DEFAULT_FS


def pick(df, *cands):
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None

def col_P(df): return pick(df, "prdM", "prDM", "prM", "prSM")
def col_T(df): return pick(df, "tv290C", "t090C", "tv290", "t190C")
def cond_Sm(df):
    c = pick(df, "c0S/m", "cond0S/m", "c0mS/cm", "c0ms/cm")
    if c is None: return None
    x = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
    return x * 0.1 if "ms/cm" in c.lower() else x


def descent_onset(pressure, fs):
    """step01's motion-based onset: last near-surface scan before the clean descent."""
    p = np.asarray(pressure, float)
    if p.size == 0 or np.isnan(p).all(): return 0
    imax = int(np.nanargmax(p)); win = max(int(round(fs)), 1)
    if imax < win + 1: return 0
    seg = pd.Series(p[:imax + 1]).rolling(win, center=True, min_periods=1).median().to_numpy()
    fwd = np.full(len(seg), np.nan); fwd[:len(seg) - win] = seg[win:] - seg[:len(seg) - win]
    descending = np.where(np.isnan(fwd), True, fwd > SECTION_SOAK_MIN_DROP_DB)
    i = imax
    while i > 0 and descending[i - 1]: i -= 1
    start = i
    span = (p[imax] - p[start]) if imax > start else 0.0
    if (imax - start) < int(5 * fs) or span < 10.0:
        below = np.where(p[:imax + 1] < 8.0)[0]
        start = int(below[-1]) if below.size else 0
    return start


def soak_and_nearsurf(path):
    """Return a dict of metrics for one cast, or None if it cannot be scored."""
    df = parse_cnv_raw(path)
    if df is None: return None
    pcol, tcol = col_P(df), col_T(df); cS = cond_Sm(df)
    if pcol is None or tcol is None or cS is None: return None
    P = pd.to_numeric(df[pcol], errors="coerce").to_numpy(float)
    T = pd.to_numeric(df[tcol], errors="coerce").to_numpy(float)
    if np.isfinite(P).sum() < 50: return None
    fs = _sample_rate(path)
    onset = descent_onset(P, fs)
    # soak dwell: in-water scans (0.2..SOAK_MAX_DB) before the clean descent
    pre = P[:onset]
    soak_scans = int(np.sum(np.isfinite(pre) & (pre > 0.2) & (pre <= SOAK_MAX_DB)))
    soak_s = soak_scans / fs
    # near-surface window of the clean descent
    p0 = P[onset] if np.isfinite(P[onset]) else np.nanmin(P)
    desc = np.arange(onset, len(P))
    mask = (P[onset:] >= p0) & (P[onset:] <= p0 + NEARSURF_DB) & np.isfinite(P[onset:])
    sel = desc[mask]
    out = dict(soak_s=soak_s, p_onset=float(p0), n_scans=int(np.isfinite(P).sum()),
               spike_S=np.nan, spike_T=np.nan, n_nearsurf=int(sel.size))
    if sel.size >= MIN_DESC_SCANS:
        Ssel = practical_salinity(cS[sel], T[sel], P[sel])
        def resid_std(x):
            s = pd.Series(np.asarray(x, float))
            return float(np.nanstd((s - s.rolling(SPIKE_WIN, center=True, min_periods=1).median()).to_numpy()))
        out["spike_S"] = resid_std(Ssel)
        out["spike_T"] = resid_std(T[sel])
    return out


print("ctd_lib:", _HAVE_LIB, "| scipy:", _HAVE_SCIPY)

## 2. Score every cast in both cruises

In [ ]:
rows = []
for k, (cruise, label) in enumerate(CRUISES):
    CRUISE_COLOR[label] = _PAL[k % len(_PAL)]
    l1 = CTD_ROOT / "cruises" / cruise / "L1" / "CTD"
    casts = sorted(l1.glob("*.cnv"))
    if not casts:
        print(f"{cruise}: no L1 cnv found in {l1}"); continue
    n_ok = 0
    for cnv in casts:
        m = soak_and_nearsurf(cnv)
        if m is None: continue
        m.update(cruise=cruise, label=label, cast=cnv.stem)
        rows.append(m); n_ok += 1
    print(f"{cruise} ({label}): scored {n_ok} of {len(casts)} casts")

D = pd.DataFrame(rows)
if not D.empty:
    out_csv = CTD_ROOT / "soak_quality_metrics.csv"
    D.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("\nper-cast metrics ->", out_csv)
    med = D.groupby("label").agg(casts=("cast", "size"),
                                 soak_s_median=("soak_s", "median"),
                                 spikeS_median=("spike_S", "median"),
                                 spikeT_median=("spike_T", "median")).round(4)
    display(med)
else:
    print("No casts scored - check CTD_ROOT and that L1 cnvs exist for both cruises.")

## 3. Figures and statistics

In [ ]:
def _box_strip(ax, groups, labels, ylabel, title, logy=False):
    rng = np.random.default_rng(0)
    data = [np.asarray(g, float)[np.isfinite(g)] for g in groups]
    bp = ax.boxplot(data, widths=0.5, showfliers=False, patch_artist=True,
                    medianprops=dict(color="black", lw=1.4))
    for i, (box, lab) in enumerate(zip(bp["boxes"], labels)):
        box.set(facecolor=CRUISE_COLOR.get(lab, "#999999"), alpha=0.35, edgecolor="#555555")
        x = rng.normal(i + 1, 0.055, size=len(data[i]))
        ax.scatter(x, data[i], s=14, color=CRUISE_COLOR.get(lab, "#555555"), alpha=0.8, edgecolors="none", zorder=3)
    ax.set_xticks(range(1, len(labels) + 1)); ax.set_xticklabels(labels)
    ax.set_ylabel(ylabel); ax.set_title(title)
    if logy: ax.set_yscale("log")
    _despine(ax)


if not D.empty:
    labels = [lab for _, lab in CRUISES if lab in set(D["label"])]
    g_soak  = [D.loc[D["label"] == l, "soak_s"].to_numpy() for l in labels]
    g_spS   = [D.loc[D["label"] == l, "spike_S"].to_numpy() for l in labels]
    g_spT   = [D.loc[D["label"] == l, "spike_T"].to_numpy() for l in labels]

    fig, axes = new_static_fig(1, 3, figsize=(12.5, 4.4))
    _box_strip(axes[0], g_soak, labels, "soak time (s)", "Soak time by cruise")
    _box_strip(axes[1], g_spS, labels, "near-surface salinity spiking (PSU)", "Top 10 db salinity noise")
    _box_strip(axes[2], g_spT, labels, "near-surface temperature spiking (degC)", "Top 10 db temperature noise")
    fig.suptitle("Soak time and near-surface data quality, by cruise", fontsize=12, y=1.00)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    finish(fig, CTD_ROOT / "soak_quality_by_cruise.png")

    # pooled: soak time vs near-surface salinity spiking (the per-cast relationship)
    fig2, ax = new_static_fig(figsize=(6.6, 5.0))
    for l in labels:
        d = D[D["label"] == l]
        ax.scatter(d["soak_s"], d["spike_S"], s=28, color=CRUISE_COLOR.get(l, "#555"),
                   alpha=0.85, edgecolors="none", label=f"{l} (n={len(d)})")
    ax.set_xlabel("soak time (s)"); ax.set_ylabel("near-surface salinity spiking (PSU)")
    ax.set_title("Per-cast: longer soak vs quieter near-surface?")
    sm = D[["soak_s", "spike_S"]].dropna()
    if _HAVE_SCIPY and len(sm) >= 5:
        rho, pr = spearmanr(sm["soak_s"], sm["spike_S"])
        ax.annotate(f"Spearman rho = {rho:.2f}, p = {pr:.3f}  (pooled, n={len(sm)})",
                    (0.5, 0.97), xycoords="axes fraction", ha="center", va="top", fontsize=8.5, color="#333")
    ax.legend(fontsize=8); _despine(ax)
    fig2.tight_layout()
    finish(fig2, CTD_ROOT / "soak_vs_nearsurf_scatter.png")

    # non-parametric tests between cruises (two groups only)
    print("\nBetween-cruise comparison (Mann-Whitney U, two-sided):")
    if _HAVE_SCIPY and len(labels) == 2:
        for name, key in (("soak time", "soak_s"), ("salinity spiking", "spike_S"), ("temperature spiking", "spike_T")):
            a = D.loc[D["label"] == labels[0], key].dropna(); b = D.loc[D["label"] == labels[1], key].dropna()
            if len(a) >= 3 and len(b) >= 3:
                u, p = mannwhitneyu(a, b, alternative="two-sided")
                print(f"  {name:<22} {labels[0]} median {a.median():.4g} vs {labels[1]} median {b.median():.4g}  (p = {p:.3f})")
    elif not _HAVE_SCIPY:
        print("  scipy not installed - medians only (pip install scipy for the tests).")
    print("\nReminder: observational, cross-cruise. Soak time and sea state are tangled; read as association, not proof.")

## 4. How to read this

- **Soak time by cruise** is the factual hook: it shows, from the data, whether each cruise actually
  held the package at the surface. If P45_06 soaked and P45_07 did not, that difference is real and
  not a matter of opinion.
- **Near-surface spiking by cruise** is the quality effect. Lower is cleaner. If the soak cruise sits
  lower, that is consistent with the soak helping, but see the caveats.
- **The scatter (pooled)** is the strongest single view: each point is one cast, both cruises together.
  A downward trend (longer soak, lower spiking) is a per-cast association that does not depend on the
  two cruises being otherwise identical. The Spearman rho and p quantify it.

**What this can and cannot say.** It can show whether soaking and near-surface quality move together
across your casts. It cannot prove the soak *caused* the difference, because rougher weather tends to
produce both shorter soaks and more near-surface disturbance, so the two are confounded. State it as:
"across P45_06 and P45_07, casts with longer surface soak tended to have quieter near-surface salinity,
consistent with the pump priming and sensors equilibrating during the soak (standard Sea-Bird / GO-SHIP
practice). A controlled test would alternate soak and no-soak casts within one cruise."

One more caution on the scatter: if soak time is effectively bimodal (one cruise soaked, the other did not), the scatter and the cruise comparison carry the *same* information, the two clusters are just the two cruises. The per-cast relationship becomes genuinely independent evidence only when there is spread in soak time *within* a cruise (some casts soaked longer than others) and the quieter near-surface tracks it. So look for a gradient, not only two clouds.

To strengthen it later: recompute the spiking on aligned + cell-thermal-mass data (removes the part of
the near-surface salinity spike that is just the conductivity/temperature time offset), and add the
fraction of loop-edit flags in the top 10 db.